# EloSense v2: PGN Move-Level Features

Does move-level game data (opening, length, material swings) predict skill better than metadata alone? The v1 notebook found metadata-only accuracy drops to 39% without the opponent's rating band. This notebook tests whether real move data closes that gap.

## 1. Setup and PGN Sample

In [ ]:
import io

import chess
import chess.pgn
import pandas as pd

DATA_PATH = "../data/club_games_data.csv"
SAMPLE_SIZE = 3000

df_full = pd.read_csv(DATA_PATH)
sample = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
sample.shape

In [ ]:
# Proof of concept: parse a single PGN and pull basic info out of it
game = chess.pgn.read_game(io.StringIO(sample["pgn"].iloc[0]))

print("ECO:", game.headers.get("ECO"))
print("White:", game.headers.get("White"), "Black:", game.headers.get("Black"))
print("Result:", game.headers.get("Result"))

moves = list(game.mainline_moves())
print("ply count:", len(moves))
print("first 10 moves:", moves[:10])

## 2. PGN Parsing Function

In [ ]:
def parse_pgn(pgn_text):
    if not isinstance(pgn_text, str):
        return None
    try:
        return chess.pgn.read_game(io.StringIO(pgn_text))
    except Exception as e:
        print(f"Failed to parse PGN: {e}")
        return None

sample["game"] = sample["pgn"].apply(parse_pgn)
fail_count = sample["game"].isna().sum()
print(f"{fail_count} of {len(sample)} PGNs failed to parse")

0 failures on this 3,000-game sample. `parse_pgn` still guards against non-string values and malformed PGNs (logs and returns `None` instead of crashing), since the full 60K+ dataset may hit edge cases this sample didn't.

## 3. Opening Extraction

In [ ]:
def get_eco(game):
    if game is None:
        return None
    return game.headers.get("ECO")

def get_opening_moves(game, n=10):
    if game is None:
        return None
    board = game.board()
    san_moves = []
    for move in list(game.mainline_moves())[:n]:
        san_moves.append(board.san(move))
        board.push(move)
    return " ".join(san_moves)

sample["eco"] = sample["game"].apply(get_eco)
sample["opening_moves"] = sample["game"].apply(get_opening_moves)

print("missing ECO:", sample["eco"].isna().sum(), "of", len(sample))
sample[["eco", "opening_moves"]].head()

ECO code is missing on 36 of 3,000 games (1.2%), those PGNs just don't have an `[ECO]` header. Using ECO as the categorical opening feature since it's already a standard classification, `opening_moves` (first 10 SAN moves) is kept as a readable backup and for sanity-checking games by eye later.

## 4. Game Length Feature

In [ ]:
def get_ply_count(game):
    if game is None:
        return None
    return len(list(game.mainline_moves()))

sample["ply_count"] = sample["game"].apply(get_ply_count)

RATING_BINS = [0, 1000, 1400, 1800, 5000]
RATING_LABELS = ["<1000", "1000-1400", "1400-1800", "1800+"]
sample["white_band"] = pd.cut(sample["white_rating"], bins=RATING_BINS, labels=RATING_LABELS)

print(sample["ply_count"].describe())
sample.groupby("white_band", observed=True)["ply_count"].mean()

Average ply count rises with rating band: ~51 under 1000, up to ~71 at 1800+. This is the opposite of the naive guess that weaker players lose fast to blunders, games actually get longer as players get stronger, likely because stronger players resist longer and don't hang mate as quickly. This alone is a promising signal for the model.

## 5. Material-Swing Feature

In [ ]:
PIECE_VALUES = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}

def material_balance(board):
    balance = 0
    for piece_type, value in PIECE_VALUES.items():
        balance += value * (len(board.pieces(piece_type, chess.WHITE)) - len(board.pieces(piece_type, chess.BLACK)))
    return balance

def material_swing_features(game):
    if game is None:
        return None, None
    board = game.board()
    balances = [0]
    for move in game.mainline_moves():
        board.push(move)
        balances.append(material_balance(board))
    max_swing = max(balances) - min(balances)

    lead_changes = 0
    prev_sign = 0
    for b in balances:
        sign = 1 if b > 0 else (-1 if b < 0 else 0)
        if sign != 0 and prev_sign != 0 and sign != prev_sign:
            lead_changes += 1
        if sign != 0:
            prev_sign = sign
    return max_swing, lead_changes

swing_results = sample["game"].apply(material_swing_features)
sample["max_swing"] = swing_results.apply(lambda t: t[0])
sample["lead_changes"] = swing_results.apply(lambda t: t[1])

print(sample[["max_swing", "lead_changes"]].describe())
sample.groupby("white_band", observed=True)[["max_swing", "lead_changes"]].mean()

`max_swing` (biggest material gap reached) is flat across rating bands (~11.4-11.6), not a useful signal on its own. `lead_changes` (how many times material advantage flips sides) climbs steadily with rating: 2.18 at <1000 up to 3.68 at 1800+. Stronger players fight back and change the material lead more often instead of a game staying one-sided after an early blunder.

## 6. Clock/Timing Feature

In [ ]:
import re

has_clk = sample["pgn"].str.contains("%clk", na=False)
print("games with %clk annotations:", has_clk.sum(), "of", len(sample))

98.1% of games have `%clk` annotations, so this feature is worth extracting.

In [ ]:
CLK_RE = re.compile(r"%clk (\d+):(\d+):(\d+(?:\.\d+)?)")

def clk_to_seconds(comment):
    m = CLK_RE.search(comment or "")
    if not m:
        return None
    h, mnt, s = m.groups()
    return int(h) * 3600 + int(mnt) * 60 + float(s)

def parse_time_control(tc):
    if not isinstance(tc, str):
        return None, 0
    parts = tc.split("+")
    try:
        base = float(parts[0])
    except ValueError:
        return None, 0
    inc = float(parts[1]) if len(parts) > 1 else 0
    return base, inc

def white_avg_move_time(game, time_control):
    if game is None:
        return None
    base, inc = parse_time_control(time_control)
    if base is None:
        return None
    clocks = []
    node = game
    ply = 0
    while node.variations:
        node = node.variations[0]
        if ply % 2 == 0:
            clocks.append(clk_to_seconds(node.comment))
        ply += 1
    clocks = [c for c in clocks if c is not None]
    if len(clocks) < 2:
        return None
    diffs = []
    prev = base
    for c in clocks:
        diffs.append(prev - c + inc)
        prev = c
    return sum(diffs) / len(diffs)

sample["white_avg_move_time"] = sample.apply(
    lambda r: white_avg_move_time(r["game"], r["time_control"]), axis=1
)

print(sample["white_avg_move_time"].describe())
sample.groupby("white_band", observed=True)["white_avg_move_time"].mean()

Average seconds spent per move (white side, including increment) drops as rating rises: 6.3s at <1000 down to 3.7s at 1800+. Stronger players spend less time per move on average, likely more pattern recognition and less calculation for routine positions. 132 of 3,000 games had unusable clocks (couldn't parse `TimeControl` or too few clock samples) and were left as `None`.

## 7. Run Full Extraction on Subsample

Combining all the feature functions above into one pass per game (one board walk instead of several) and sanity-checking the output against the raw PGN.

In [ ]:
def extract_all_features(pgn_text, time_control):
    if not isinstance(pgn_text, str):
        return None
    try:
        game = chess.pgn.read_game(io.StringIO(pgn_text))
    except Exception:
        return None
    if game is None:
        return None

    eco = game.headers.get("ECO")
    board = game.board()
    balances = [0]
    white_clocks = []
    opening_moves = []
    ply = 0
    node = game
    base, inc = parse_time_control(time_control)

    while node.variations:
        node = node.variations[0]
        move = node.move
        if ply < 10:
            opening_moves.append(board.san(move))
        board.push(move)
        balances.append(material_balance(board))
        if ply % 2 == 0:
            white_clocks.append(clk_to_seconds(node.comment))
        ply += 1

    max_swing = max(balances) - min(balances)
    lead_changes = 0
    prev_sign = 0
    for b in balances:
        sign = 1 if b > 0 else (-1 if b < 0 else 0)
        if sign != 0 and prev_sign != 0 and sign != prev_sign:
            lead_changes += 1
        if sign != 0:
            prev_sign = sign

    white_clocks_clean = [c for c in white_clocks if c is not None]
    avg_move_time = None
    if base is not None and len(white_clocks_clean) >= 2:
        diffs = []
        prev = base
        for c in white_clocks_clean:
            diffs.append(prev - c + inc)
            prev = c
        avg_move_time = sum(diffs) / len(diffs)

    return pd.Series({
        "eco": eco,
        "opening_moves": " ".join(opening_moves),
        "ply_count": ply,
        "max_swing": max_swing,
        "lead_changes": lead_changes,
        "white_avg_move_time": avg_move_time,
    })

extracted = sample.apply(lambda r: extract_all_features(r["pgn"], r["time_control"]), axis=1)
extracted.head()

In [ ]:
# Manual sanity check: compare extracted features against a few raw games
for i in [0, 1, 2]:
    print("row", i)
    print("  raw time_control:", sample["time_control"].iloc[i], "| raw result:", sample["white_result"].iloc[i])
    print("  extracted:", extracted.iloc[i].to_dict())
    print()

Checked rows 0-2 by eye against the raw PGN headers: ECO codes match (`C24`, `C44`, `C23`), and `white_avg_move_time` is consistent with each game's `time_control` (faster average on the 180+2 bullet game than the 900+10 rapid game). 0 extraction failures on the full 3,000-game subsample.

## 8. Benchmark Extraction Speed

In [ ]:
import time

start = time.time()
_ = sample.apply(lambda r: extract_all_features(r["pgn"], r["time_control"]), axis=1)
elapsed = time.time() - start

rate = len(sample) / elapsed
full_size = len(df_full)
est_full_seconds = full_size / rate

print(f"{len(sample)} games in {elapsed:.2f}s -> {rate:.1f} games/sec")
print(f"full dataset ({full_size} games) estimated at {est_full_seconds:.0f}s (~{est_full_seconds/60:.1f} min)")

About 540 games/sec, so the full 66,879-game dataset is estimated at roughly 2 minutes. That's cheap enough to run on the full dataset rather than capping at a larger-but-limited subsample.